# AA test tutorial 
AA test is important part of randomized controlled experiment, for example AB test. 

The objectives of the AA test are to verify the assumption of uniformity of samples as a result of the applied partitioning method, to select the best partition from the available ones, and to verify the applicability of statistical criteria for checking uniformity. 

For example, there is a hypothesis about the absence of dependence of features on each other. If this hypothesis is not followed, the AA test will fail.

[Wiki AA test](https://github.com/sb-ai-lab/HypEx/wiki/%D0%90%D0%90-Test) with more detailed description of terms for AA test.

<ul>
  <li><a href="#creation-of-a-new-test-dataset-with-synthetic-data">Creation of a new test dataset with synthetic data.
  <li><a href="#one-split-of-aa-test">One split of AA test.
  <li><a href="#aa-test">AA test.
  <li><a href="#aa-test-with-stratification">AA test with stratification.
</ul>

In [5]:
from hypex import AATest
from hypex.dataset import (
    ConstGroupRole,
    Dataset,
    InfoRole,
    StratificationRole,
    TargetRole,
    TreatmentRole,
)
from hypex.utils import create_test_data

## Creation of a new test dataset with synthetic data. 

In order to be able to work with our data in HypEx, first we need to convert it into `dataset`. It is important to mark the data fields by assigning the appropriate `roles`:
- TargetRole: a role for columns that contain features or predictor variables. Our split will be based on them. Applied by default if the role is not specified for the column.
- TreatmentRole: a role for columns that show the treatment or intervention.
- InfoRole: a role for columns that contain information about the data, such as user IDs. 

In [6]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
    }, data=create_test_data(),
)
data

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0.0,8.0,1.0,481.5,469.111111,27.0,M,E-commerce
1,1.0,0.0,0.0,509.0,421.444444,48.0,M,Logistics
2,2.0,0.0,0.0,468.5,410.888889,21.0,M,E-commerce
3,3.0,0.0,0.0,480.0,429.666667,35.0,F,Logistics
4,4.0,7.0,1.0,505.0,490.111111,55.0,F,Logistics
...,...,...,...,...,...,...,...,...
9995,9995.0,0.0,0.0,529.0,434.444444,46.0,M,E-commerce
9996,9996.0,0.0,0.0,465.0,424.0,48.0,F,Logistics
9997,9997.0,0.0,0.0,462.0,427.0,50.0,M,E-commerce
9998,9998.0,10.0,1.0,473.0,435.333333,49.0,M,E-commerce


In [7]:
data.roles

{'user_id': Info(<class 'int'>),
 'pre_spends': Target(<class 'float'>),
 'post_spends': Target(<class 'float'>),
 'gender': Stratification(<class 'str'>),
 'signup_month': Default(<class 'float'>),
 'treat': Default(<class 'float'>),
 'age': Default(<class 'float'>),
 'industry': Default(<class 'object'>)}

## AA test
Then we run the experiment on our prepared dataset, wrapped into ExperimentData. In this case we select one of the pre-assembled pipeline, AA_TEST.
We can set the number of iterations for simple execution. In this case the random states are the numbers of each iteration.

In [8]:
test = AATest(n_iterations=10)
result = test.execute(data)

2026-08-27 19:18:52 | INFO     | hypex.experiment | ▶ Process started: ParamsExperiment [pandas]
  0%|          | 0/10 [00:00<?, ?it/s]2026-08-27 19:18:52 | INFO     | hypex.experiment | ▶ Process started: NaDropper [pandas]
2026-08-27 19:18:52 | INFO     | hypex.experiment | ✓ Process finished: NaDropper in 0.005s
2026-08-27 19:18:52 | INFO     | hypex.experiment | ▶ Process started: AASplitter [pandas]
2026-08-27 19:18:52 | INFO     | hypex.experiment | ✓ Process finished: AASplitter in 0.044s
2026-08-27 19:18:52 | INFO     | hypex.experiment | ▶ Process started: Experiment [pandas]
2026-08-27 19:18:52 | INFO     | hypex.experiment | ▶ Process started: GroupSizes [pandas]
2026-08-27 19:18:52 | INFO     | hypex.experiment | ✓ Process finished: GroupSizes in 0.018s
2026-08-27 19:18:52 | INFO     | hypex.experiment | ▶ Process started: OnRoleExperiment [pandas]
2026-08-27 19:18:52 | INFO     | hypex.experiment | ▶ Process started: GroupDifference [pandas]
2026-08-27 19:18:52 | INFO     

In [9]:
result.resume

,feature,group,TTest aa test,KSTest aa test,TTest best split,KSTest best split,result,control mean,test mean,difference,difference %
0,post_spends,test,OK,OK,NOT OK,NOT OK,NOT OK,None,None,None,None
1,pre_spends,test,OK,OK,NOT OK,NOT OK,NOT OK,None,None,None,None


**Interpretation of AA test results**

Each row in the table corresponds to a target feature being tested for equality between the control and test groups. Two statistical tests are used:

- **TTest**: tests if means are statistically different.
- **KSTest**: tests if distributions differ.

The `OK` / `NOT OK` labels show whether the difference is statistically significant. A `NOT OK` result indicates a possible imbalance.

Typical threshold:
- If p-value < 0.05 → `NOT OK` (statistically significant difference)
- If p-value ≥ 0.05 → `OK` (no significant difference)

If any metric has a `NOT OK` status in the `AA test` column, it means at least one iteration showed significant difference.


In [10]:
result.aa_score

,score,pass
pre_spends TTest test,0.95,True
post_spends TTest test,0.95,True
pre_spends KSTest test,0.95,True
post_spends KSTest test,0.95,True
mean TTest all,0.95,True
mean KSTest all,0.95,True


**Interpreting `aa_score`**

This output shows p-values and the overall pass/fail status for each test type and feature. A high p-value (close to 1.0) means the test passed — the groups are similar.

- `score`: p-value of the statistical test.
- `pass`: True if no iterations showed significant differences.

Note: Even if the average p-value is high, the `pass` might still be False if at least one of the iterations had a p-value < 0.05.


In [11]:
result.best_split

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry,AASplitter┴rs 5┴best,split
0,0.0,8.0,1.0,481.5,469.111111,27.0,M,E-commerce,test,test
1,1.0,0.0,0.0,509.0,421.444444,48.0,M,Logistics,test,test
2,2.0,0.0,0.0,468.5,410.888889,21.0,M,E-commerce,test,test
3,3.0,0.0,0.0,480.0,429.666667,35.0,F,Logistics,test,test
4,4.0,7.0,1.0,505.0,490.111111,55.0,F,Logistics,test,test
...,...,...,...,...,...,...,...,...,...,...
9995,9995.0,0.0,0.0,529.0,434.444444,46.0,M,E-commerce,test,test
9996,9996.0,0.0,0.0,465.0,424.0,48.0,F,Logistics,control,control
9997,9997.0,0.0,0.0,462.0,427.0,50.0,M,E-commerce,control,control
9998,9998.0,10.0,1.0,473.0,435.333333,49.0,M,E-commerce,test,test


**About `best_split`**

This shows the best found split of the dataset, where control and test groups are as similar as possible in terms of target metrics.

You can use this split for future modeling or as a validation check before proceeding to actual experiments.


In [12]:
result.best_split_statistic

,splitter_id,pre_spends┆stats GroupDifference mean┆pre_spends control,pre_spends┆stats GroupDifference mean┆pre_spends test,post_spends┆stats GroupDifference mean┆post_spends control,post_spends┆stats GroupDifference mean┆post_spends test,"['pre_spends', 'post_spends'] GroupDifference control mean test┆pre_spends","['pre_spends', 'post_spends'] GroupDifference test mean test┆pre_spends","['pre_spends', 'post_spends'] GroupDifference difference test┆pre_spends","['pre_spends', 'post_spends'] GroupDifference difference % test┆pre_spends","['pre_spends', 'post_spends'] GroupDifference control mean test┆post_spends",...,post_spends TTest pass test,pre_spends KSTest p-value test,pre_spends KSTest pass test,post_spends KSTest p-value test,post_spends KSTest pass test,mean TTest p-value all,mean TTest pass all,mean KSTest p-value all,mean KSTest pass all,mean test score
5,AASplitter┴rs 5┴,487.516364,487.646238,451.930635,451.951576,487.516364,487.646238,0.129874,0.02664,451.930635,...,0.0,0.844867,0.0,0.734671,0.0,0.862752,0.0,0.789769,0.0,0.814097


**Understanding `best_split_statistic`**

This table contains detailed statistics for the best (most balanced) split found across all iterations. You can compare:

- Mean values in control vs test group.
- Absolute and relative differences.
- p-values for both tests.

Ideally, all rows should have `OK` in both TTest and KSTest columns, and small difference values (<1%).

In [13]:
result.experiments

,splitter_id,pre_spends TTest p-value test,pre_spends TTest pass test,post_spends TTest p-value test,post_spends TTest pass test,pre_spends KSTest p-value test,pre_spends KSTest pass test,post_spends KSTest p-value test,post_spends KSTest pass test,mean TTest p-value all,mean TTest pass all,mean KSTest p-value all,mean KSTest pass all,mean test score
0,AASplitter┴rs 0┴,0.844592,0.0,0.610746,0.0,0.923080,0.0,0.858959,0.0,0.727669,0.0,0.891020,0.0,0.836570
1,AASplitter┴rs 1┴,0.838412,0.0,0.395390,0.0,0.536387,0.0,0.454649,0.0,0.616901,0.0,0.495518,0.0,0.535979
2,AASplitter┴rs 2┴,0.704171,0.0,0.237212,0.0,0.792391,0.0,0.387478,0.0,0.470692,0.0,0.589935,0.0,0.550187
3,AASplitter┴rs 3┴,0.740471,0.0,0.543427,0.0,0.626651,0.0,0.645020,0.0,0.641949,0.0,0.635835,0.0,0.637873
4,AASplitter┴rs 4┴,0.358317,0.0,0.572201,0.0,0.947768,0.0,0.859580,0.0,0.465259,0.0,0.903674,0.0,0.757536
5,AASplitter┴rs 5┴,0.745601,0.0,0.979903,0.0,0.844867,0.0,0.734671,0.0,0.862752,0.0,0.789769,0.0,0.814097
6,AASplitter┴rs 6┴,0.513281,0.0,0.180916,0.0,0.543694,0.0,0.123011,0.0,0.347098,0.0,0.333353,0.0,0.337934
7,AASplitter┴rs 7┴,0.310064,0.0,0.971270,0.0,0.093735,0.0,0.838520,0.0,0.640667,0.0,0.466127,0.0,0.524307
8,AASplitter┴rs 8┴,0.696455,0.0,0.127011,0.0,0.690389,0.0,0.333427,0.0,0.411733,0.0,0.511908,0.0,0.478516
9,AASplitter┴rs 9┴,0.332867,0.0,0.811954,0.0,0.196866,0.0,0.103440,0.0,0.572410,0.0,0.150153,0.0,0.290905


# AA Test with random states

We can also adjust some of the preset parameters of the experiment by assigning them to the respective params of the experiment. I.e. here we set the range of the random states we want to run our AA test for. 

In [14]:
test = AATest(random_states=[56, 72, 2, 43])
result = test.execute(data)

2026-08-27 19:19:07 | INFO     | hypex.experiment | ▶ Process started: ParamsExperiment [pandas]
  0%|          | 0/4 [00:00<?, ?it/s]2026-08-27 19:19:07 | INFO     | hypex.experiment | ▶ Process started: NaDropper [pandas]
2026-08-27 19:19:07 | INFO     | hypex.experiment | ✓ Process finished: NaDropper in 0.005s
2026-08-27 19:19:07 | INFO     | hypex.experiment | ▶ Process started: AASplitter [pandas]
2026-08-27 19:19:07 | INFO     | hypex.experiment | ✓ Process finished: AASplitter in 0.043s
2026-08-27 19:19:07 | INFO     | hypex.experiment | ▶ Process started: Experiment [pandas]
2026-08-27 19:19:07 | INFO     | hypex.experiment | ▶ Process started: GroupSizes [pandas]
2026-08-27 19:19:07 | INFO     | hypex.experiment | ✓ Process finished: GroupSizes in 0.016s
2026-08-27 19:19:07 | INFO     | hypex.experiment | ▶ Process started: OnRoleExperiment [pandas]
2026-08-27 19:19:07 | INFO     | hypex.experiment | ▶ Process started: GroupDifference [pandas]
2026-08-27 19:19:07 | INFO     |

In [15]:
result.resume

,feature,group,TTest aa test,KSTest aa test,TTest best split,KSTest best split,result,control mean,test mean,difference,difference %
0,post_spends,test,OK,OK,NOT OK,NOT OK,NOT OK,None,None,None,None
1,pre_spends,test,OK,OK,NOT OK,NOT OK,NOT OK,None,None,None,None


In [16]:
result.aa_score

,score,pass
pre_spends TTest test,0.95,True
post_spends TTest test,0.95,True
pre_spends KSTest test,0.95,True
post_spends KSTest test,0.95,True
mean TTest all,0.95,True
mean KSTest all,0.95,True


In [17]:
result.best_split

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry,AASplitter┴rs 5┴best,AASplitter┴rs 2┴best,split
0,0.0,8.0,1.0,481.5,469.111111,27.0,M,E-commerce,test,test,test
1,1.0,0.0,0.0,509.0,421.444444,48.0,M,Logistics,test,test,test
2,2.0,0.0,0.0,468.5,410.888889,21.0,M,E-commerce,test,control,control
3,3.0,0.0,0.0,480.0,429.666667,35.0,F,Logistics,test,test,test
4,4.0,7.0,1.0,505.0,490.111111,55.0,F,Logistics,test,control,control
...,...,...,...,...,...,...,...,...,...,...,...
9995,9995.0,0.0,0.0,529.0,434.444444,46.0,M,E-commerce,test,control,control
9996,9996.0,0.0,0.0,465.0,424.0,48.0,F,Logistics,control,test,test
9997,9997.0,0.0,0.0,462.0,427.0,50.0,M,E-commerce,control,control,control
9998,9998.0,10.0,1.0,473.0,435.333333,49.0,M,E-commerce,test,test,test


In [18]:
result.best_split_statistic

,splitter_id,pre_spends┆stats GroupDifference mean┆pre_spends control,pre_spends┆stats GroupDifference mean┆pre_spends test,post_spends┆stats GroupDifference mean┆post_spends control,post_spends┆stats GroupDifference mean┆post_spends test,"['pre_spends', 'post_spends'] GroupDifference control mean test┆pre_spends","['pre_spends', 'post_spends'] GroupDifference test mean test┆pre_spends","['pre_spends', 'post_spends'] GroupDifference difference test┆pre_spends","['pre_spends', 'post_spends'] GroupDifference difference % test┆pre_spends","['pre_spends', 'post_spends'] GroupDifference control mean test┆post_spends",...,post_spends TTest pass test,pre_spends KSTest p-value test,pre_spends KSTest pass test,post_spends KSTest p-value test,post_spends KSTest pass test,mean TTest p-value all,mean TTest pass all,mean KSTest p-value all,mean KSTest pass all,mean test score
2,AASplitter┴rs 2┴,487.505825,487.657794,451.454996,452.437706,487.505825,487.657794,0.151969,0.031173,451.454996,...,0.0,0.792391,0.0,0.387478,0.0,0.470692,0.0,0.589935,0.0,0.550187


In [19]:
result.experiments

,splitter_id,pre_spends TTest p-value test,pre_spends TTest pass test,post_spends TTest p-value test,post_spends TTest pass test,pre_spends KSTest p-value test,pre_spends KSTest pass test,post_spends KSTest p-value test,post_spends KSTest pass test,mean TTest p-value all,mean TTest pass all,mean KSTest p-value all,mean KSTest pass all,mean test score
0,AASplitter┴rs 56┴,0.097514,0.0,0.234211,0.0,0.023608,1.0,0.153932,0.0,0.165862,0.0,0.088770,0.5,0.114468
1,AASplitter┴rs 72┴,0.149202,0.0,0.174721,0.0,0.326334,0.0,0.199106,0.0,0.161962,0.0,0.262720,0.0,0.229134
2,AASplitter┴rs 2┴,0.704171,0.0,0.237212,0.0,0.792391,0.0,0.387478,0.0,0.470692,0.0,0.589935,0.0,0.550187
3,AASplitter┴rs 43┴,0.032293,1.0,0.621038,0.0,0.003871,1.0,0.921811,0.0,0.326666,0.5,0.462841,0.5,0.417449


# AA Test with stratification

Depending on your requirements it is possible to stratify the data. You can set `stratification=True` and `StratificationRole` in `Dataset` to run it with stratification.

Stratified AA tests ensure that both groups (control/test) have the same proportions of categories (e.g. same % of genders or regions). This prevents imbalances in categorical features that can distort results.

Make sure to assign `StratificationRole` to relevant columns in your dataset before enabling stratification.

In [20]:
test = AATest(random_states=[56, 72, 2, 43], stratification=True)
result = test.execute(data)

2026-08-27 19:19:11 | INFO     | hypex.experiment | ▶ Process started: ParamsExperiment [pandas]
  0%|          | 0/4 [00:00<?, ?it/s]2026-08-27 19:19:11 | INFO     | hypex.experiment | ▶ Process started: NaDropper [pandas]
2026-08-27 19:19:11 | INFO     | hypex.experiment | ✓ Process finished: NaDropper in 0.005s
2026-08-27 19:19:11 | INFO     | hypex.experiment | ▶ Process started: AASplitterWithStratification [pandas]
2026-08-27 19:19:11 | INFO     | hypex.experiment | ✓ Process finished: AASplitterWithStratification in 0.047s
2026-08-27 19:19:11 | INFO     | hypex.experiment | ▶ Process started: Experiment [pandas]
2026-08-27 19:19:11 | INFO     | hypex.experiment | ▶ Process started: GroupSizes [pandas]
2026-08-27 19:19:11 | INFO     | hypex.experiment | ✓ Process finished: GroupSizes in 0.014s
2026-08-27 19:19:11 | INFO     | hypex.experiment | ▶ Process started: OnRoleExperiment [pandas]
2026-08-27 19:19:11 | INFO     | hypex.experiment | ▶ Process started: GroupDifference [pand

In [21]:
result.resume

,feature,group,TTest aa test,KSTest aa test,TTest best split,KSTest best split,result,control mean,test mean,difference,difference %
0,post_spends,test,OK,OK,NOT OK,NOT OK,NOT OK,None,None,None,None
1,pre_spends,test,OK,OK,NOT OK,NOT OK,NOT OK,None,None,None,None


In [22]:
result.aa_score

,score,pass
pre_spends TTest test,0.95,True
post_spends TTest test,0.95,True
pre_spends KSTest test,0.95,True
post_spends KSTest test,0.95,True
mean TTest all,0.95,True
mean KSTest all,0.95,True


In [23]:
result.best_split

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry,AASplitter┴rs 5┴best,AASplitter┴rs 2┴best,AASplitterWithStratification┴rs 2┴best,split
0,0.0,8.0,1.0,481.5,469.111111,27.0,M,E-commerce,test,test,test,test
1,1.0,0.0,0.0,509.0,421.444444,48.0,M,Logistics,test,test,test,test
2,2.0,0.0,0.0,468.5,410.888889,21.0,M,E-commerce,test,control,control,control
3,3.0,0.0,0.0,480.0,429.666667,35.0,F,Logistics,test,test,test,test
4,4.0,7.0,1.0,505.0,490.111111,55.0,F,Logistics,test,control,control,control
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9995.0,0.0,0.0,529.0,434.444444,46.0,M,E-commerce,test,control,control,control
9996,9996.0,0.0,0.0,465.0,424.0,48.0,F,Logistics,control,test,test,test
9997,9997.0,0.0,0.0,462.0,427.0,50.0,M,E-commerce,control,control,control,control
9998,9998.0,10.0,1.0,473.0,435.333333,49.0,M,E-commerce,test,test,test,test


In [24]:
result.best_split_statistic

,splitter_id,pre_spends┆stats GroupDifference mean┆pre_spends control,pre_spends┆stats GroupDifference mean┆pre_spends test,post_spends┆stats GroupDifference mean┆post_spends control,post_spends┆stats GroupDifference mean┆post_spends test,"['pre_spends', 'post_spends'] GroupDifference control mean test┆pre_spends","['pre_spends', 'post_spends'] GroupDifference test mean test┆pre_spends","['pre_spends', 'post_spends'] GroupDifference difference test┆pre_spends","['pre_spends', 'post_spends'] GroupDifference difference % test┆pre_spends","['pre_spends', 'post_spends'] GroupDifference control mean test┆post_spends",...,post_spends TTest pass test,pre_spends KSTest p-value test,pre_spends KSTest pass test,post_spends KSTest p-value test,post_spends KSTest pass test,mean TTest p-value all,mean TTest pass all,mean KSTest p-value all,mean KSTest pass all,mean test score
2,AASplitterWithStratification┴rs 2┴,487.504287,487.659367,451.470531,452.421833,487.504287,487.659367,0.15508,0.031811,451.470531,...,0.0,0.792391,0.0,0.417001,0.0,0.475474,0.0,0.604696,0.0,0.561622


In [25]:
result.experiments

,splitter_id,pre_spends TTest p-value test,pre_spends TTest pass test,post_spends TTest p-value test,post_spends TTest pass test,pre_spends KSTest p-value test,pre_spends KSTest pass test,post_spends KSTest p-value test,post_spends KSTest pass test,mean TTest p-value all,mean TTest pass all,mean KSTest p-value all,mean KSTest pass all,mean test score
0,AASplitterWithStratification┴rs 56┴,0.097514,0.0,0.234211,0.0,0.023608,1.0,0.153932,0.0,0.165862,0.0,0.088770,0.5,0.114468
1,AASplitterWithStratification┴rs 72┴,0.149202,0.0,0.174721,0.0,0.326334,0.0,0.199106,0.0,0.161962,0.0,0.262720,0.0,0.229134
2,AASplitterWithStratification┴rs 2┴,0.698409,0.0,0.252539,0.0,0.792391,0.0,0.417001,0.0,0.475474,0.0,0.604696,0.0,0.561622
3,AASplitterWithStratification┴rs 43┴,0.032293,1.0,0.621038,0.0,0.003871,1.0,0.921811,0.0,0.326666,0.5,0.462841,0.5,0.417449


# AA Test by samples 

Depending on your requirements and size of data it is possible to estimate AA test on samples the data. You can set `sample_size=size` to run it. 

In [26]:
test = AATest(n_iterations=10, sample_size=0.3)
result = test.execute(data)

2026-08-27 19:19:15 | INFO     | hypex.experiment | ▶ Process started: ParamsExperiment [pandas]
  0%|          | 0/10 [00:00<?, ?it/s]2026-08-27 19:19:15 | INFO     | hypex.experiment | ▶ Process started: NaDropper [pandas]
2026-08-27 19:19:15 | INFO     | hypex.experiment | ✓ Process finished: NaDropper in 0.005s
2026-08-27 19:19:15 | INFO     | hypex.experiment | ▶ Process started: AASplitter [pandas]
2026-08-27 19:19:16 | INFO     | hypex.experiment | ✓ Process finished: AASplitter in 0.038s
2026-08-27 19:19:16 | INFO     | hypex.experiment | ▶ Process started: Experiment [pandas]
2026-08-27 19:19:16 | INFO     | hypex.experiment | ▶ Process started: GroupSizes [pandas]
2026-08-27 19:19:16 | INFO     | hypex.experiment | ✓ Process finished: GroupSizes in 0.013s
2026-08-27 19:19:16 | INFO     | hypex.experiment | ▶ Process started: OnRoleExperiment [pandas]
2026-08-27 19:19:16 | INFO     | hypex.experiment | ▶ Process started: GroupDifference [pandas]
2026-08-27 19:19:16 | INFO     

KeyError: nan

In [27]:
result.resume

,feature,group,TTest aa test,KSTest aa test,TTest best split,KSTest best split,result,control mean,test mean,difference,difference %
0,post_spends,test,OK,OK,NOT OK,NOT OK,NOT OK,None,None,None,None
1,pre_spends,test,OK,OK,NOT OK,NOT OK,NOT OK,None,None,None,None


In [ ]:
result.aa_score

,score,pass
pre_spends TTest test_1,0.95,True
post_spends TTest test_1,0.95,True
pre_spends KSTest test_1,0.95,True
post_spends KSTest test_1,0.95,True


In [ ]:
result.best_split

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry,split
0,0.0,11.0,1.0,476.0,436.888889,28.0,F,E-commerce,control
1,1.0,1.0,1.0,519.5,525.222222,36.0,F,Logistics,control
2,2.0,0.0,0.0,498.5,414.333333,69.0,F,Logistics,test_1
3,3.0,10.0,1.0,473.0,445.888889,43.0,F,E-commerce,test_1
4,4.0,11.0,1.0,495.0,428.111111,56.0,F,E-commerce,test_1
...,...,...,...,...,...,...,...,...,...
9995,9995.0,0.0,0.0,475.0,408.111111,51.0,M,Logistics,test_1
9996,9996.0,0.0,0.0,472.5,414.666667,22.0,F,E-commerce,test_1
9997,9997.0,0.0,0.0,474.0,419.222222,63.0,M,E-commerce,test_1
9998,9998.0,4.0,1.0,481.0,519.888889,21.0,F,Logistics,test_1


In [ ]:
result.best_split_statistic

,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value,KSTest pass,KSTest p-value
0,pre_spends,test_1,486.8862275449102,487.53776908023485,0.6515415353246681,0.13381802533418696,OK,0.2510028127854083,OK,None
1,post_spends,test_1,452.0774284763806,452.235485975212,0.15805749883139697,0.034962484051481724,OK,0.8930513511282197,OK,None


In [ ]:
result.experiments

,splitter_id,pre_spends GroupDifference control mean test_1,pre_spends GroupDifference test mean test_1,pre_spends GroupDifference difference test_1,pre_spends GroupDifference difference % test_1,post_spends GroupDifference control mean test_1,post_spends GroupDifference test mean test_1,post_spends GroupDifference difference test_1,post_spends GroupDifference difference % test_1,pre_spends TTest p-value test_1,...,post_spends TTest pass test_1,pre_spends KSTest p-value test_1,pre_spends KSTest pass test_1,post_spends KSTest p-value test_1,post_spends KSTest pass test_1,mean TTest p-value,mean TTest pass,mean KSTest p-value,mean KSTest pass,mean test score
0,AASplitter┴rs 0┴,487.492604,487.431952,-0.060652,-0.012442,450.998685,452.426490,1.427805,0.316587,0.914487,...,False,NaN,False,NaN,False,0.568377,0.0,0,0.0,0.189459
1,AASplitter┴rs 1┴,487.911853,487.359220,-0.552633,-0.113265,453.169792,452.045528,-1.124264,-0.248089,0.330682,...,False,NaN,False,NaN,False,0.335019,0.0,0,0.0,0.111673
2,AASplitter┴rs 2┴,487.263529,487.472360,0.208832,0.042858,453.065398,452.061582,-1.003817,-0.221561,0.711837,...,False,NaN,False,NaN,False,0.551563,0.0,0,0.0,0.183854
3,AASplitter┴rs 3┴,486.886228,487.537769,0.651542,0.133818,452.077428,452.235486,0.158057,0.034962,0.251003,...,False,NaN,False,NaN,False,0.572027,0.0,0,0.0,0.190676
4,AASplitter┴rs 4┴,486.762325,487.561764,0.799439,0.164236,452.709590,452.123542,-0.586048,-0.129453,0.156058,...,False,NaN,False,NaN,False,0.385857,0.0,0,0.0,0.128619
5,AASplitter┴rs 5┴,487.136667,487.494772,0.358105,0.073512,451.589053,452.321948,0.732894,0.162292,0.526323,...,False,NaN,False,NaN,False,0.528788,0.0,0,0.0,0.176263
6,AASplitter┴rs 6┴,486.904303,487.535607,0.631304,0.129657,452.432245,452.173236,-0.259009,-0.057248,0.264264,...,False,NaN,False,NaN,False,0.544629,0.0,0,0.0,0.181543
7,AASplitter┴rs 7┴,487.354074,487.456411,0.102337,0.020998,453.439671,451.995411,-1.444260,-0.318512,0.856311,...,False,NaN,False,NaN,False,0.536788,0.0,0,0.0,0.178929
8,AASplitter┴rs 8┴,486.892884,487.536525,0.643641,0.132194,451.726758,452.296533,0.569775,0.126133,0.256943,...,False,NaN,False,NaN,False,0.442485,0.0,0,0.0,0.147495
9,AASplitter┴rs 9┴,488.483570,487.258875,-1.224695,-0.250714,454.938262,451.735593,-3.202670,-0.703979,0.030778,...,True,NaN,False,NaN,False,0.018582,1.0,0,0.0,0.006194


# AATest with Target Role for a categorical feature

It is possible to assign Target Role to categorical features. A categorical feature can also be the target or outcome variable. In this case, the Chi-square test is added to the pipeline of AATest.

In [ ]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "treat": TreatmentRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": TargetRole(str)
    }, data=create_test_data(),
)
data

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0.0,2.0,1.0,507.0,514.111111,59.0,M,E-commerce
1,1.0,8.0,1.0,501.5,450.777778,69.0,M,Logistics
2,2.0,0.0,0.0,496.0,424.222222,45.0,M,E-commerce
3,3.0,0.0,0.0,461.0,441.444444,51.0,M,E-commerce
4,4.0,0.0,0.0,489.0,410.444444,35.0,M,Logistics
...,...,...,...,...,...,...,...,...
9995,9995.0,7.0,1.0,477.5,467.000000,27.0,M,E-commerce
9996,9996.0,0.0,0.0,455.5,426.888889,47.0,M,E-commerce
9997,9997.0,6.0,1.0,473.0,482.444444,20.0,M,E-commerce
9998,9998.0,4.0,1.0,489.5,499.333333,60.0,F,Logistics


In [ ]:
test = AATest(n_iterations=10)
result = test.execute(data)

100%|██████████| 10/10 [00:04<00:00,  2.26it/s]


In [ ]:
result.resume

,feature,group,TTest aa test,KSTest aa test,Chi2Test aa test,TTest best split,KSTest best split,Chi2Test best split,result,control mean,test mean,difference,difference %
0,pre_spends,test_1,OK,OK,NaN,OK,OK,NaN,OK,487.356000,487.460231,0.104231,0.021387
1,post_spends,test_1,OK,OK,NaN,OK,OK,NaN,OK,451.664938,452.415809,0.750871,0.166245
2,gender,test_1,NaN,NaN,OK,NaN,NaN,OK,OK,NaN,NaN,NaN,NaN


In [ ]:
result.aa_score

,score,pass
pre_spends TTest test_1,0.95,True
post_spends TTest test_1,0.95,True
pre_spends KSTest test_1,0.95,True
post_spends KSTest test_1,0.95,True
gender Chi2Test test_1,0.95,True


In [ ]:
result.best_split

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry,split
0,0.0,2.0,1.0,507.0,514.111111,59.0,M,E-commerce,control
1,1.0,8.0,1.0,501.5,450.777778,69.0,M,Logistics,control
2,2.0,0.0,0.0,496.0,424.222222,45.0,M,E-commerce,test_1
3,3.0,0.0,0.0,461.0,441.444444,51.0,M,E-commerce,test_1
4,4.0,0.0,0.0,489.0,410.444444,35.0,M,Logistics,test_1
...,...,...,...,...,...,...,...,...,...
9995,9995.0,7.0,1.0,477.5,467.000000,27.0,M,E-commerce,test_1
9996,9996.0,0.0,0.0,455.5,426.888889,47.0,M,E-commerce,test_1
9997,9997.0,6.0,1.0,473.0,482.444444,20.0,M,E-commerce,test_1
9998,9998.0,4.0,1.0,489.5,499.333333,60.0,F,Logistics,test_1


In [ ]:
result.best_split_statistic

,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value,KSTest pass,KSTest p-value,Chi2Test pass,Chi2Test p-value
0,pre_spends,test_1,487.356,487.4602310597645,0.10423105976451552,0.021387047612941856,OK,0.7960766529784996,OK,NaN,NaN,NaN
1,post_spends,test_1,451.664938271605,452.4158088326051,0.7508705610000561,0.16624504082016767,OK,0.36839695002856443,OK,NaN,NaN,NaN
2,gender,test_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OK,1.0


In [ ]:
result.experiments

,splitter_id,pre_spends GroupDifference control mean test_1,pre_spends GroupDifference test mean test_1,pre_spends GroupDifference difference test_1,pre_spends GroupDifference difference % test_1,post_spends GroupDifference control mean test_1,post_spends GroupDifference test mean test_1,post_spends GroupDifference difference test_1,post_spends GroupDifference difference % test_1,pre_spends TTest p-value test_1,...,post_spends KSTest pass test_1,gender Chi2Test p-value test_1,gender Chi2Test pass test_1,mean TTest p-value,mean TTest pass,mean KSTest p-value,mean KSTest pass,mean Chi2Test p-value,mean Chi2Test pass,mean test score
0,AASplitter┴rs 0┴,487.286493,487.529399,0.242906,0.049849,451.798052,452.282080,0.484028,0.107134,0.547002,...,False,0.272458,False,0.554520,0.0,0,0.0,0.272458,0.0,0.219887
1,AASplitter┴rs 1┴,487.528141,487.287433,-0.240708,-0.049373,452.549573,451.528421,-1.021151,-0.225644,0.550636,...,False,0.269199,False,0.385931,0.0,0,0.0,0.269199,0.0,0.184866
2,AASplitter┴rs 2┴,487.489227,487.326962,-0.162265,-0.033286,451.582062,452.499074,0.917012,0.203066,0.687450,...,False,0.759602,False,0.479715,0.0,0,0.0,0.759602,0.0,0.399784
3,AASplitter┴rs 3┴,487.356000,487.460231,0.104231,0.021387,451.664938,452.415809,0.750871,0.166245,0.796077,...,False,1.000000,False,0.582237,0.0,0,0.0,1.000000,0.0,0.516447
4,AASplitter┴rs 4┴,487.586055,487.227680,-0.358375,-0.073500,451.678043,452.407896,0.729854,0.161587,0.374249,...,False,0.347871,False,0.378107,0.0,0,0.0,0.347871,0.0,0.214770
5,AASplitter┴rs 5┴,487.189579,487.627589,0.438010,0.089905,451.538162,452.544793,1.006631,0.222934,0.277469,...,False,0.486647,False,0.252667,0.0,0,0.0,0.486647,0.0,0.245192
6,AASplitter┴rs 6┴,487.502770,487.312946,-0.189824,-0.038938,451.985228,452.095910,0.110682,0.024488,0.637894,...,False,0.268135,False,0.766208,0.0,0,0.0,0.268135,0.0,0.260496
7,AASplitter┴rs 7┴,487.615016,487.202921,-0.412095,-0.084512,452.044177,452.036685,-0.007492,-0.001657,0.306896,...,False,0.552795,False,0.649868,0.0,0,0.0,0.552795,0.0,0.351092
8,AASplitter┴rs 8┴,486.937290,487.873233,0.935943,0.192210,451.540825,452.533937,0.993112,0.219938,0.020295,...,False,0.964712,False,0.127237,0.5,0,0.0,0.964712,0.0,0.411332
9,AASplitter┴rs 9┴,487.308289,487.508465,0.200176,0.041078,451.348084,452.736294,1.388210,0.307570,0.619674,...,False,0.360739,False,0.357989,0.0,0,0.0,0.360739,0.0,0.215893


# AATest with unequal group sizes

AATest can be performed to get a split with unequal the groups of different sizes by using `unequal_size` argument. Also Whelch correction can be applied by adding `t_test_equal_vat=False` argument while initiating AATest instance.

In [ ]:
test = AATest(n_iterations=10, control_size=0.3, t_test_equal_var=False)
result = test.execute(data)

  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:04<00:00,  2.25it/s]


In [ ]:
result.best_split.data.groupby("split").agg("count")

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
split,,,,,,,,
control,2731,2731,2731,2731,2731,2731,2731,2731
test_1,6270,6270,6270,6270,6270,6270,6270,6270


In [ ]:
result.best_split_statistic

,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value,KSTest pass,KSTest p-value,Chi2Test pass,Chi2Test p-value
0,pre_spends,test_1,487.5825704870011,487.3321371610845,-0.25043332591656053,-0.05136223915190863,OK,0.5639126299904302,OK,NaN,NaN,NaN
1,post_spends,test_1,451.7230969526831,452.1786283891547,0.45553143647163097,0.10084306946991362,OK,0.6168334001746815,OK,NaN,NaN,NaN
2,gender,test_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OK,1.0


# AAnTest

AAnTest is an extension of AATest that allows to split the dataset into several test groups, additionally to the control group.

In [ ]:
test = AATest(groups_sizes=[0.3, 0.2, 0.2, 0.3])
result = test.execute(data)

100%|██████████| 10/10 [00:08<00:00,  1.22it/s]


In [ ]:
result.best_split.data.groupby("split").agg("count")

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
split,,,,,,,,
control,2710,2710,2710,2710,2710,2710,2710,2710
test_1,1792,1792,1792,1792,1792,1792,1792,1792
test_2,1802,1802,1802,1802,1802,1802,1802,1802
test_3,2697,2697,2697,2697,2697,2697,2697,2697


In [ ]:
result.best_split_statistic

,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value,KSTest pass,KSTest p-value,Chi2Test pass,Chi2Test p-value
0,pre_spends,test_1,487.51162361623614,487.45535714285717,-0.05626647337896884,-0.011541565503936368,OK,0.9237459969594596,OK,NaN,NaN,NaN
1,pre_spends,test_2,487.51162361623614,487.23473917869035,-0.27688443754578884,-0.05679545350979476,OK,0.6346033057388143,OK,NaN,NaN,NaN
2,pre_spends,test_3,487.51162361623614,487.38857990359656,-0.12304371263957137,-0.02523913414143042,OK,0.8113756749318759,OK,NaN,NaN,NaN
3,post_spends,test_1,451.11914719147194,452.28211805555554,1.1629708640836043,0.25779683068738457,OK,0.32943324267093954,OK,NaN,NaN,NaN
4,post_spends,test_2,451.11914719147194,453.4614009125663,2.34225372109438,0.5192095559846122,OK,0.05446016907853474,OK,NaN,NaN,NaN
5,post_spends,test_3,451.11914719147194,451.8560952498661,0.7369480583941481,0.16335995999774422,OK,0.49088418106719045,OK,NaN,NaN,NaN
6,gender,test_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OK,0.8717290989479588
7,gender,test_2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OK,0.8911496337110267
8,gender,test_3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OK,0.7053738742058828


# AATest with partially pre-defined groups

Certain users can be pre-assigned to either the test or the control group, so that they are not randomly assigned. This can be done using the `ConstGroupRole` role. In order to pre-assign users to the control group they should have a value of `control`, and in the test group they should have a value of `test` in the column with the role `ConstGroupRole`. Users that are not pre-assigned to either the control or the test group should have `None`, so that they will be assigned randomly.

In [ ]:
pd_data= create_test_data()
pd_data.loc[pd_data["treat"]==0, "const_grp"] = "control"
pd_data.loc[pd_data["treat"]==1, "const_grp"] = "test"
pd_data.loc[2000:, "const_grp"] = None

data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "const_grp": ConstGroupRole(str),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
        "industry": TargetRole(str),
    }, data=pd_data,
)
data

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry,const_grp
0,0.0,0.0,0.0,498.0,405.111111,30.0,M,E-commerce,control
1,1.0,0.0,0.0,494.0,416.000000,68.0,F,Logistics,control
2,2.0,10.0,1.0,469.0,437.777778,25.0,F,Logistics,test
3,3.0,0.0,0.0,442.0,414.333333,68.0,F,Logistics,control
4,4.0,0.0,0.0,483.0,418.333333,35.0,M,E-commerce,control
...,...,...,...,...,...,...,...,...,...
9995,9995.0,6.0,1.0,479.0,505.888889,55.0,F,Logistics,None
9996,9996.0,5.0,1.0,516.5,499.333333,56.0,F,Logistics,None
9997,9997.0,3.0,1.0,489.5,526.000000,61.0,M,Logistics,None
9998,9998.0,0.0,0.0,468.0,434.222222,52.0,M,E-commerce,None


In [ ]:
test = AATest(n_iterations=1)
result = test.execute(data)

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  1.41it/s]


In [ ]:
result.resume

,feature,group,TTest aa test,KSTest aa test,Chi2Test aa test,TTest best split,KSTest best split,Chi2Test best split,result,control mean,test mean,difference,difference %
0,pre_spends,test_1,OK,OK,NaN,OK,OK,NaN,OK,486.906311,487.295794,0.389482,0.079991
1,post_spends,test_1,NOT OK,OK,NaN,NOT OK,OK,NaN,OK,445.172017,458.400095,13.228078,2.971453
2,industry,test_1,NaN,NaN,OK,NaN,NaN,OK,OK,NaN,NaN,NaN,NaN


In [ ]:
result.best_split

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry,const_grp,split
0,0.0,0.0,0.0,498.0,405.111111,30.0,M,E-commerce,control,control
1,1.0,0.0,0.0,494.0,416.000000,68.0,F,Logistics,control,control
2,2.0,10.0,1.0,469.0,437.777778,25.0,F,Logistics,test,test_1
3,3.0,0.0,0.0,442.0,414.333333,68.0,F,Logistics,control,control
4,4.0,0.0,0.0,483.0,418.333333,35.0,M,E-commerce,control,control
...,...,...,...,...,...,...,...,...,...,...
9995,9995.0,6.0,1.0,479.0,505.888889,55.0,F,Logistics,None,control
9996,9996.0,5.0,1.0,516.5,499.333333,56.0,F,Logistics,None,control
9997,9997.0,3.0,1.0,489.5,526.000000,61.0,M,Logistics,None,test_1
9998,9998.0,0.0,0.0,468.0,434.222222,52.0,M,E-commerce,None,control


## Common issues and tips

- **Missing roles**: Make sure all target variables are assigned `TargetRole`. Columns without roles may cause silent failure.
- **Stratification**: If your dataset contains categorical features (e.g. `gender`, `region`) that may affect the outcome, use `StratificationRole` and enable `stratification=True` in `AATest(...)`.
- **Imbalanced categories**: If some categories have too few samples, stratified splits may become unstable. Consider filtering or merging rare categories.
- **Random fluctuations**: On small datasets, it's normal to see occasional `NOT OK` results. Use more iterations (e.g. `n_iterations=50`) for stability.
- **Missing values**: NaNs in stratification columns may be treated as separate categories. Clean or fill missing values before stratified AA tests.